# 08_step5_threshold_matrix

Step 5: decision-boundary thresholds. For at-risk individuals we compute the
minimum BMI reduction needed to bring predicted hypertension risk down to the
cohort median risk line (a practical, underwriting-oriented target rather than
an absolute 0.5 cutoff, which is unreachable for a low-incidence outcome).

The result is an age x baseline-BMI matrix of required BMI reductions
(Table 8, Figure 5). It also reports feature importances (Table 9) and the
share of at-risk individuals for whom the target is unreachable even within a
15% reduction cap - the age-dominated group with no feasible recourse, echoing
notebook 03.

In [1]:
# 08_step5_threshold_matrix.ipynb
# Reverse-engineer the minimum BMI reduction to reach a target risk line.

import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split

ROOT = os.path.abspath("..")
DATA = os.path.join(ROOT, "data")
FIG  = os.path.join(ROOT, "results", "figures")
TAB  = os.path.join(ROOT, "results", "tables")

htn = pd.read_parquet(os.path.join(DATA, "htn_analysis.parquet"))
htn["female"] = (htn["SEX"] == 2).astype(float)
FEATS = ["BMI", "age", "female", "smoke_cur", "exer_reg", "walk_days"]
X = htn[FEATS].astype(float); y = htn["incident"].astype(int)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

clf = GradientBoostingClassifier(n_estimators=200, max_depth=3,
                                 random_state=42).fit(Xtr, ytr)
allp = clf.predict_proba(X)[:, 1]
target_risk = np.percentile(allp, 50)   # cohort median risk as the target line
print(f"Target risk line (median): {target_risk:.4f}")

Target risk line (median): 0.0226


In [2]:
# Minimum single-variable BMI reduction to reach the target risk line.
def min_bmi_to_target(row, target, floor=18.5, step=0.1):
    x = row[FEATS].values.astype(float).copy(); b0 = x[0]
    for d in np.arange(0, b0 - floor + step, step):
        x[0] = b0 - d
        if clf.predict_proba(x.reshape(1, -1))[0, 1] <= target:
            return round(d, 1)
    return None   # unreachable within floor (age-dominated)

Xall = X.copy(); Xall["p"] = allp
thr70 = np.percentile(allp, 70)              # top-30% risk = loading candidates
hi = Xall[Xall["p"] >= thr70].copy()
hi["minBMIred"] = hi.apply(lambda r: min_bmi_to_target(r, target_risk), axis=1)
print(f"Loading candidates (top 30%): {len(hi)}")
print(f"Unreachable within floor (age-dominated): "
      f"{hi['minBMIred'].isna().mean()*100:.1f}%")

Loading candidates (top 30%): 9354
Unreachable within floor (age-dominated): 55.9%


In [3]:
# Age x baseline-BMI threshold matrix (Table 8) and heatmap (Figure 5).
hi["age_grp"] = pd.cut(hi["age"], [19,45,55,65,75,120],
                       labels=["19-44","45-54","55-64","65-74","75+"])
hi["bmi_grp"] = pd.cut(hi["BMI"], [0,23,25,27.5,30,100],
                       labels=["<23","23-25","25-27.5","27.5-30","30+"])
mat = hi.pivot_table(index="bmi_grp", columns="age_grp",
                     values="minBMIred", aggfunc="mean", observed=True)
mat.round(2).to_csv(os.path.join(TAB, "table8_threshold_matrix.csv"))
hi.to_parquet(os.path.join(DATA, "step5_thresholds.parquet"))
print(mat.round(1).to_string())

sns.set_theme(style="white", context="paper")
plt.rcParams.update({"font.size": 11, "savefig.dpi": 600})
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(mat, annot=True, fmt=".1f", cmap="Greys", linewidths=0.5,
            linecolor="0.8", vmin=0, vmax=8, annot_kws={"size": 10},
            cbar_kws={"label": "Required BMI reduction (kg/m2)"}, ax=ax)
ax.set_xlabel("Age group"); ax.set_ylabel("Baseline BMI group")
fig.savefig(os.path.join(FIG, "fig5_threshold_matrix.png"), dpi=600, bbox_inches="tight")
fig.savefig(os.path.join(FIG, "fig5_threshold_matrix.pdf"), bbox_inches="tight")
plt.close(fig)
print("Figure 5 saved (png + pdf).")

age_grp  19-44  45-54  55-64  65-74  75+
bmi_grp                                 
<23        NaN    1.4    1.7    1.9  1.7
23-25      NaN    3.1    3.5    3.7  3.4
25-27.5    0.3    2.4    4.7    5.3  4.5
27.5-30    0.8    3.4    7.0    7.2  5.4
30+        1.4    5.8    8.1    6.5  7.9


Figure 5 saved (png + pdf).


In [4]:
# Feature importances underlying the boundary (Table 9).
imp = pd.Series(clf.feature_importances_, index=FEATS).sort_values(ascending=False)
imp.round(4).to_csv(os.path.join(TAB, "table9_feature_importance.csv"),
                    header=["importance"])
print((imp * 100).round(1).to_string())
print("\nBMI (47%) and age (43%) dominate the boundary; behavioural variables"
      " contribute little, consistent with the risk-model odds ratios.")

BMI          47.0
age          42.5
walk_days     5.2
exer_reg      2.3
female        2.1
smoke_cur     0.8

BMI (47%) and age (43%) dominate the boundary; behavioural variables contribute little, consistent with the risk-model odds ratios.
